# Classification model training

Here we will begin the classification model training (with binary data) for which we first need to establish the dataset for it to be used in tensorflow.

## Preparing the dataset

### Importing necessary libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
from tensorflow.keras import Model, layers

from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

import keras
import os

2025-06-24 11:48:58.253615: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 11:48:58.472798: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 11:48:58.474719: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-24 11:49:01.848396: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### Setting parameters

In [ ]:
# Data parameters
im_width = 256
im_height = im_width
n_classes = 5

# Training Parameters
learning_rate = 0.001
epochs = 30
batch_size = 32
s_per_epoch = 20
last_dense = 32
print(f"Batch size: {batch_size}")
print(f"Units of the last Dense Layer: {last_dense}")

### Reading the data

In [ ]:
features_folders = sorted(os.listdir("patrones_binary/"))

# First iteration through folders to know total number of pathces
patchs_total_number = 0
for f_folder in features_folders:
    patchs_total_number += len(
        next(os.walk("patrones_binary/%s/" % (f_folder)))[2]
    )  # list of names all images in the given path

images = np.zeros((patchs_total_number, im_height, im_width), dtype=np.float32)
labels = np.zeros((patchs_total_number), dtype=np.float32)

n = 0
for folder in features_folders:
    names = next(os.walk("patrones_binary/%s/" % (folder)))[2]  # list of names all images in the given path

    for name in names:
        # Load features
        feature = np.loadtxt("patrones_binary/%s/%s" % (folder, name))

        # Load labels
        label = features_folders.index(folder)  # (hexagonos, hexagonos vacios, homogenea, rayas, vacia) = (0, 1, 2, 3, 4)

        images[n] = feature
        labels[n] = label

        n += 1

### Separating in train/test/validation

In [ ]:
images_trainval, images_test, labels_trainval, labels_test = train_test_split(
    images[:, :, :],
    labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

images_train, images_val, labels_train, labels_val = train_test_split(
    images_trainval[:, :, :],
    labels_trainval,
    test_size=(1/8),
    random_state=42,
    shuffle=True,
)

labels_train = to_categorical(labels_train, num_classes=n_classes)
labels_test = to_categorical(labels_test, num_classes=n_classes)
labels_val = to_categorical(labels_val, num_classes=n_classes)

### Calculamos weights
total_points = labels.shape[0]

class_weights = {
    0: total_points / np.sum([labels == 0]),
    1: total_points / np.sum([labels == 1]),
    2: total_points / np.sum([labels == 2]),
    3: total_points / np.sum([labels == 3]),
    4: total_points / np.sum([labels == 4]),
}

# sample weights
sample_weights = np.ones((images_train.shape[0], im_width, im_height, n_classes))

sample_weights[:, :, :, 0] = class_weights[0]
sample_weights[:, :, :, 1] = class_weights[1]
sample_weights[:, :, :, 2] = class_weights[2]
sample_weights[:, :, :, 3] = class_weights[3]
sample_weights[:, :, :, 4] = class_weights[4]


[=================================================================================================== ] 99%

### Model training

In [ ]:
# Define model (https://www.tensorflow.org/tutorials/images/cnn?hl=es-419)

model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu', input_shape=(256, 256, 1)),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
  tf.keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), activation='relu'),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(units = last_dense, activation='relu'),
  tf.keras.layers.Dense(5)
])

2025-06-20 11:24:51.152669: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [ ]:
# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.CategoricalAccuracy(),
             tf.keras.metrics.OneHotMeanIoU(num_classes=n_classes),
             ],
    loss_weights=list(class_weights.values()),
)

In [ ]:
model.summary()

In [ ]:
# Train model
result = model.fit(
        x=images_train,
        y=labels_train,
        validation_data=(images_val, labels_val),
        batch_size=batch_size,
        epochs=epochs,
        steps_per_epoch=s_per_epoch
    )


In [ ]:
i = 1
model_path = f"model_C_{i}.h5"
model.save(model_path)

In [ ]:
# Print results
loss_test, cat_accuracy_test, onehotmIoU_test = model.evaluate(images_test, labels_test, batch_size=1000)

print(f"Loss test: {loss_test}")
print(f"Categorical accuracy: {cat_accuracy_test}")
print(f"One hot mean IoU: {onehotmIoU_test}")

# AUC score
predictions = model.predict(images_test)
y_score_train = model.predict(images_train)

micro_roc_auc_ovr_test = roc_auc_score( labels_test, predictions,
multi_class="ovr",
average="micro",
)
micro_roc_auc_ovr_train = roc_auc_score( labels_train, y_score_train,
multi_class="ovr",
average="micro",
)

print(f"Micro roc auc test: {micro_roc_auc_ovr_test}")
print(f"Micro roc auc train: {micro_roc_auc_ovr_train}")

In [ ]:
### Compute the confusion matrix
# Convert predictions to class labels
predicted_classes = np.argmax(predictions, axis=1)

# Convert true labels to class labels
true_classes = np.argmax(labels_test, axis=1)

# Compute confusion matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)

# Store confusion matrix in pandas DataFrame
confusion_df = pd.DataFrame(conf_matrix, 
                            index=[f'Class {i}' for i in range(n_classes)],
                            columns=[f'Predicted {i}' for i in range(n_classes)])

# (hexagonos, hexagonos vacios, homogenea, rayas, vacia) = (0, 1, 2, 3, 4)
# Print confusion matrix
print("Confusion Matrix:")
print(confusion_df)

In [ ]:
i = 1

results = pd.DataFrame(result.history)

results.to_csv(f'results_model_C_{i}.csv', index=False)